# SSR System Demo

This notebook demonstrates the complete SSR (Semantic Similarity Rating) pipeline:
1. Create persona
2. Generate LLM response
3. Rate response using SSR
4. Test with different questions

In [1]:
import sys
from pathlib import Path
import os
from dotenv import load_dotenv

# Add src to path
sys.path.append(str(Path.cwd().parent))

# Load environment variables
load_dotenv(Path.cwd().parent / '.env')

from src.persona.generator import PersonaGenerator
from src.llm.client import LLMClient
from src.llm.prompts import PromptBuilder
from src.ssr.embeddings import EmbeddingService
from src.ssr.rating_engine import RatingEngine
from src.scales.registry import create_default_scales

import warnings
warnings.filterwarnings('ignore')

## 1. Initialize Components

In [2]:
# Initialize all components
print("Initializing components...")

# Persona generator
persona_gen = PersonaGenerator(random_seed=42)
print("✓ Persona generator")

# LLM client
llm_client = LLMClient(model="gpt-4o", temperature=0.5)
print("✓ LLM client (GPT-4o)")

# Embedding service
embedding_service = EmbeddingService(model_id="text-embedding-3-small")
print("✓ Embedding service")

# Scale registry
scale_registry = create_default_scales()
print(f"✓ Scale registry ({len(scale_registry.scales)} scales)")

# Rating engine
rating_engine = RatingEngine(scale_registry, embedding_service)
print("✓ Rating engine")

print("\n✅ All components initialized!")

Initializing components...
✓ Persona generator
✓ LLM client (GPT-4o)
✓ Embedding service
✓ Scale registry (13 scales)
✓ Rating engine

✅ All components initialized!


## 2. Generate Test Persona

In [3]:
# Generate a persona
persona = persona_gen.generate_persona()

print("Generated Persona:")
print("=" * 80)
print(f"ID: {persona.id}")
print(f"\nDemographics:")
for key, value in persona.demographics.items():
    print(f"   {key}: {value}")
print(f"\nPsychographics:")
for key, value in persona.psychographics.items():
    print(f"   {key}: {value}")

print(f"\nNatural Language Description:")
print(f"   {persona.to_description()}")

Generated Persona:
ID: P00001

Demographics:
   gender: Female
   age: 26
   occupation: Finance
   age_band: 18-35

Psychographics:
   category_buyer: ['Lottery played in-store/person', 'Paper scratchcards bought in person', 'Lottery played online/app']
   sc_players: SC Players
   inertia: 5
   target_group: Young non-rejectors 18-35

Natural Language Description:
   26 years old, female, works in Finance, buys lottery played in-store/person, paper scratchcards bought in person, lottery played online/app, likes to try new and different products


## 3. Define Sample Concept

In [4]:
# Sample concept description
concept_description = """Christmas AR Message Scratchcard

Price: 250 CZK

Occasion: Christmas

Features:
- Guaranteed win scratchcard with QR code
- Record a personal Christmas video message with festive AR filters
- Recipient scans QR code to see your video message appear in AR above the card
- Fun and interactive way to send Christmas greetings
- Available in stores and online
"""

print("Test Concept:")
print("=" * 80)
print(concept_description)

Test Concept:
Christmas AR Message Scratchcard

Price: 250 CZK

Occasion: Christmas

Features:
- Guaranteed win scratchcard with QR code
- Record a personal Christmas video message with festive AR filters
- Recipient scans QR code to see your video message appear in AR above the card
- Fun and interactive way to send Christmas greetings
- Available in stores and online



## 4. Test Purchase Intent (Primary Metric)

In [5]:
# Build prompts
system_prompt = PromptBuilder.build_system_prompt(persona)
user_prompt = PromptBuilder.build_purchase_intent_prompt(concept_description)

print("System Prompt (Persona Conditioning):")
print("=" * 80)
print(system_prompt)
print("\n\nUser Prompt (Question):")
print("=" * 80)
print(user_prompt)

System Prompt (Persona Conditioning):
You are a consumer participating in a market research survey about scratchcard products.

Your profile: You are 26 years old, female, works in Finance, buys lottery played in-store/person, paper scratchcards bought in person, lottery played online/app, likes to try new and different products.

When answering questions:
- Respond naturally and authentically as this specific person would
- Base your opinions on your demographic and psychographic characteristics
- Be honest and specific in your responses
- Vary your language naturally (don't be repetitive)
- Express genuine opinions, both positive and negative
- Keep responses concise (1-3 sentences typically)

Remember: You are not an AI assistant - you are a real consumer with real opinions about products.


User Prompt (Question):
Here is a scratchcard product concept:

Christmas AR Message Scratchcard

Price: 250 CZK

Occasion: Christmas

Features:
- Guaranteed win scratchcard with QR code
- Recor

In [6]:
# Generate LLM response
print("\nGenerating LLM response...")
response_text = llm_client.generate_response(system_prompt, user_prompt)

print("\nLLM Response:")
print("=" * 80)
print(response_text)


Generating LLM response...

LLM Response:
I would be somewhat likely to buy this scratchcard as a unique and fun way to send Christmas greetings. The idea of incorporating AR and a personal video message is appealing and adds a personal touch that traditional cards lack. However, the price seems a bit steep for a scratchcard, so I'd have to consider it as more of a special gift for someone close rather than a casual purchase.


In [7]:
# Rate using SSR
print("\nRating response using SSR...")
result = rating_engine.rate_answer(
    answer_text=response_text,
    scale_id="likert5_purchase_intent_v1",
    temperature=0.5,
    return_debug_info=True
)

print("\nSSR Rating Result:")
print("=" * 80)
print(f"Chosen Level: {result.chosen_level_value}")
print(f"Label: {result.chosen_level_label}")
print(f"Formatted: {result.to_formatted_response()}")
print(f"\nConfidence: {result.probabilities[result.chosen_level_index]:.3f}")

print(f"\nProbability Distribution:")
scale = scale_registry.get_scale("likert5_purchase_intent_v1")
for i, (label, prob) in enumerate(zip(scale.level_labels, result.probabilities)):
    bar = '█' * int(prob * 50)
    print(f"   {i+1}. {label:30s} {prob:.3f} {bar}")

print(f"\nRaw Similarities:")
for i, (label, sim) in enumerate(zip(scale.level_labels, result.raw_similarities)):
    print(f"   {i+1}. {label:30s} {sim:.3f}")


Rating response using SSR...

SSR Rating Result:
Chosen Level: 2
Label: Probably would
Formatted: (2) Probably would

Confidence: 0.212

Probability Distribution:
   1. Definitely would               0.203 ██████████
   2. Probably would                 0.212 ██████████
   3. Might or might not             0.190 █████████
   4. Probably would not             0.201 ██████████
   5. Definitely would not           0.195 █████████

Raw Similarities:
   1. Definitely would               0.585
   2. Probably would                 0.604
   3. Might or might not             0.550
   4. Probably would not             0.578
   5. Definitely would not           0.563


## 5. Test Other Question Types

In [8]:
# Test Uniqueness
print("\n" + "="*80)
print("UNIQUENESS")
print("="*80)

user_prompt = PromptBuilder.build_uniqueness_prompt(concept_description)
response_text = llm_client.generate_response(system_prompt, user_prompt)
print(f"\nResponse: {response_text}")

result = rating_engine.rate_answer(
    response_text,
    scale_id="likert5_uniqueness_v1",
    temperature=0.5
)
print(f"\nRating: {result.to_formatted_response()}")
print(f"Confidence: {result.probabilities[result.chosen_level_index]:.3f}")


UNIQUENESS

Response: This scratchcard concept feels quite unique compared to others I've seen. The integration of AR and a personalized video message adds a modern and interactive twist to the traditional scratchcard experience. It's a fun and innovative way to combine technology with a festive occasion, which is definitely appealing to someone like me who enjoys trying new products.

Rating: (2) Very new and different
Confidence: 0.327


In [9]:
# Test Likeability (6-point scale)
print("\n" + "="*80)
print("LIKEABILITY")
print("="*80)

user_prompt = PromptBuilder.build_likeability_prompt(concept_description)
response_text = llm_client.generate_response(system_prompt, user_prompt)
print(f"\nResponse: {response_text}")

result = rating_engine.rate_answer(
    response_text,
    scale_id="likert6_likeability_v1",
    temperature=0.5
)
print(f"\nRating: {result.to_formatted_response()}")
print(f"Confidence: {result.probabilities[result.chosen_level_index]:.3f}")


LIKEABILITY

Response: I really like the concept of the Christmas AR Message Scratchcard. It combines the fun of a guaranteed win with a personal touch through the video message, making it a unique and interactive way to send holiday greetings. Plus, the use of AR filters adds a playful, modern twist that makes it more engaging.

Rating: (2) Like very much
Confidence: 0.199


In [10]:
# Test Excitement (4-point scale)
print("\n" + "="*80)
print("EXCITEMENT")
print("="*80)

user_prompt = PromptBuilder.build_excitement_prompt(concept_description)
response_text = llm_client.generate_response(system_prompt, user_prompt)
print(f"\nResponse: {response_text}")

result = rating_engine.rate_answer(
    response_text,
    scale_id="likert4_excitement_v1",
    temperature=0.5
)
print(f"\nRating: {result.to_formatted_response()}")
print(f"Confidence: {result.probabilities[result.chosen_level_index]:.3f}")


EXCITEMENT

Response: This concept is quite exciting to me! I love the idea of combining the fun of scratchcards with the personal touch of a video message, especially with festive AR filters. It's a unique and interactive way to send Christmas greetings, and I'm always interested in trying out new and different products like this.

Rating: (1) Very exciting
Confidence: 0.311


## 6. Test Multiple Personas

In [11]:
# Generate multiple personas and test purchase intent
print("\n" + "="*80)
print("TESTING MULTIPLE PERSONAS")
print("="*80)

personas = persona_gen.generate_batch(5)
results_summary = []

for i, p in enumerate(personas, 1):
    print(f"\n--- Persona {i}: {p.to_description()} ---")
    
    # Generate response
    sys_prompt = PromptBuilder.build_system_prompt(p)
    usr_prompt = PromptBuilder.build_purchase_intent_prompt(concept_description)
    resp = llm_client.generate_response(sys_prompt, usr_prompt)
    
    # Rate
    rating = rating_engine.rate_answer(
        resp,
        scale_id="likert5_purchase_intent_v1",
        temperature=0.5
    )
    
    print(f"Response: {resp[:100]}...")
    print(f"Rating: {rating.to_formatted_response()}")
    
    results_summary.append({
        'persona': p.to_description(),
        'rating': rating.chosen_level_value,
        'label': rating.chosen_level_label,
        'confidence': rating.probabilities[rating.chosen_level_index]
    })


TESTING MULTIPLE PERSONAS

--- Persona 1: 18 years old, male, works in Government, buys none of the above ---
Response: I probably wouldn't buy the Christmas AR Message Scratchcard at 250 CZK. While the idea of sending a...
Rating: (4) Probably would not

--- Persona 2: 26 years old, female, works in Unemployed, buys paper scratchcards bought in person, likes to try new and different products ---
Response: I love the idea of combining a personal video message with a scratchcard, especially with the festiv...
Rating: (2) Probably would

--- Persona 3: 28 years old, male, works in Healthcare, buys none of the above, likes to try new and different products ---
Response: I'm not very likely to buy this scratchcard at 250 CZK. While the idea of sending a personal Christm...
Rating: (4) Probably would not

--- Persona 4: 37 years old, female, works in Student, buys lottery played online/app ---
Response: I think the Christmas AR Message Scratchcard sounds like a creative and fun way to send

In [12]:
# Summary
import pandas as pd

df_summary = pd.DataFrame(results_summary)
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(df_summary)

print(f"\nMean Rating: {df_summary['rating'].mean():.2f}")
print(f"Rating Distribution:")
print(df_summary['label'].value_counts())


SUMMARY
                                             persona  rating  \
0  18 years old, male, works in Government, buys ...       4   
1  26 years old, female, works in Unemployed, buy...       2   
2  28 years old, male, works in Healthcare, buys ...       4   
3  37 years old, female, works in Student, buys l...       2   
4  47 years old, male, works in Other, buys none ...       4   

                label  confidence  
0  Probably would not    0.210836  
1      Probably would    0.206973  
2  Probably would not    0.207647  
3      Probably would    0.209596  
4  Probably would not    0.217354  

Mean Rating: 3.20
Rating Distribution:
label
Probably would not    3
Probably would        2
Name: count, dtype: int64


## Summary

This notebook demonstrated the complete SSR pipeline:

✅ **Persona Generation** - Created realistic consumer profiles with demographics and psychographics

✅ **LLM Response Generation** - GPT-4o generated natural language responses conditioned on persona

✅ **Semantic Similarity Rating** - Mapped free-text to Likert scales using anchor embeddings

✅ **Multiple Scale Types** - Tested 4-point, 5-point, and 6-point Likert scales

✅ **Probability Distributions** - Got full probability mass functions, not just single ratings

**Key Advantages of SSR:**
- More realistic response variance than direct LLM ratings
- Transparent and reproducible (anchors are fixed)
- Richer qualitative data (free-text responses)
- Better discrimination between levels

**Next Steps:**
- Scale up to full survey with all questions
- Implement conditional logic for skip patterns
- Generate complete dataset
- Validate against ground truth